In [1]:
import os 
import requests

pdf_path = "human-nutrition-text.pdf"

if not os.path.exists(pdf_path):
    print("File does not exist, downloading...")

    url = "https://pressbooks.oer.hawaii.edu/humannutrition2/open/download?type=pdf"

    filename = pdf_path

    response = requests.get(url)

    if response.status_code == 200:
        with open(filename, "wb") as f:
            f.write(response.content)
        print(f"The file was downloaded successfully as {filename}")

    else:
        print("Failed to download the file")

In [2]:
import fitz
from tqdm.auto import tqdm

def text_formatter(text):
    cleaned_text = text.replace("\n", " ").strip()

    return cleaned_text

def open_and_read_pdf(pdf_path):
    doc = fitz.open(pdf_path)

    pages_and_texts = []

    for page_number, page in tqdm(enumerate(doc)):
        text = page.get_text()
        text = text_formatter(text)

        pages_and_texts.append({
            "page_number": page_number - 41,
            "page_char_count": len(text),
            "page_word_count": len(text.split(" ")),
            "page_sentence_count_raw": len(text.split(". ")),
            "page_token_count": len(text) / 4,
            "text": text
        })

    return pages_and_texts

pages_and_texts = open_and_read_pdf(pdf_path=pdf_path)
pages_and_texts[:2]

0it [00:00, ?it/s]

[{'page_number': -41,
  'page_char_count': 29,
  'page_word_count': 4,
  'page_sentence_count_raw': 1,
  'page_token_count': 7.25,
  'text': 'Human Nutrition: 2020 Edition'},
 {'page_number': -40,
  'page_char_count': 0,
  'page_word_count': 1,
  'page_sentence_count_raw': 1,
  'page_token_count': 0.0,
  'text': ''}]

In [3]:
import pandas as pd

df = pd.DataFrame(pages_and_texts)

df.head()

,page_number,page_char_count,page_word_count,page_sentence_count_raw,page_token_count,text
0,-41,29,4,1,7.25,Human Nutrition: 2020 Edition
1,-40,0,1,1,0.00,
2,-39,320,54,1,80.00,Human Nutrition: 2020 Edition UNIVERSITY OF ...
3,-38,212,32,1,53.00,Human Nutrition: 2020 Edition by University of...
4,-37,797,147,3,199.25,Contents Preface University of Hawai‘i at Mā...


In [4]:
df.describe().round(2)

,page_number,page_char_count,page_word_count,page_sentence_count_raw,page_token_count
count,1208.00,1208.00,1208.00,1208.00,1208.00
mean,562.50,1148.00,199.50,10.52,287.00
std,348.86,560.38,95.83,6.55,140.10
min,-41.00,0.00,1.00,1.00,0.00
25%,260.75,762.00,134.00,5.00,190.50
50%,562.50,1231.50,216.00,10.00,307.88
75%,864.25,1603.50,272.00,15.00,400.88
max,1166.00,2308.00,430.00,39.00,577.00


In [5]:
from spacy.lang.en import English

nlp = English()

nlp.add_pipe("sentencizer")

doc = nlp("This is some sentence. This is another sentence. And this is another one.")

list(doc.sents)

[This is some sentence., This is another sentence., And this is another one.]

In [6]:
pages_and_texts[0]

{'page_number': -41,
 'page_char_count': 29,
 'page_word_count': 4,
 'page_sentence_count_raw': 1,
 'page_token_count': 7.25,
 'text': 'Human Nutrition: 2020 Edition'}

In [7]:
for item in tqdm(pages_and_texts):
    item["sentences"] = list(nlp(item["text"]).sents)

    item["sentences"] = [str(sentence) for sentence in item["sentences"]]

    item["page_sentence_count_spacy"] = len(item["sentences"])


  0%|          | 0/1208 [00:00<?, ?it/s]

In [8]:
import random

random.sample(pages_and_texts, k=3)

[{'page_number': 36,
  'page_char_count': 1533,
  'page_word_count': 254,
  'page_sentence_count_raw': 25,
  'page_token_count': 383.25,
  'text': 'Experimental test. Coindet administered iodine tincture orally to  his patients with goiter.  Interpret results. Coindet’s iodine treatment was successful.    Hypothesis. French chemist Chatin proposed that the low iodine  content in food and water in certain areas far away from the ocean  was the primary cause of goiter, and renounced the theory that  goiter was the result of poor hygiene.  Experimental test. In the late 1860s the program, “The stamping- out of goiter,” started with people in several villages in France being  given iodine tablets.  Results. The program was effective and 80 percent of goitrous  children were cured.    Hypothesis. In 1918, Swiss doctor Bayard proposed iodizing salt as  a good way to treat areas endemic with goiter.  Experimental test. Iodized salt was transported by mules to a  small village at the base of t

In [9]:
df = pd.DataFrame(pages_and_texts)

df.describe().round(2)

,page_number,page_char_count,page_word_count,page_sentence_count_raw,page_token_count,page_sentence_count_spacy
count,1208.00,1208.00,1208.00,1208.00,1208.00,1208.00
mean,562.50,1148.00,199.50,10.52,287.00,10.32
std,348.86,560.38,95.83,6.55,140.10,6.30
min,-41.00,0.00,1.00,1.00,0.00,0.00
25%,260.75,762.00,134.00,5.00,190.50,5.00
50%,562.50,1231.50,216.00,10.00,307.88,10.00
75%,864.25,1603.50,272.00,15.00,400.88,15.00
max,1166.00,2308.00,430.00,39.00,577.00,28.00


In [10]:
num_sentence_chunk_size = 10

def split_list(input_list, slice_size=num_sentence_chunk_size):
    return [input_list[i:i+slice_size] for i in range(0, len(input_list), slice_size)]

test_list = list(range(29))
split_list(test_list)

[[0, 1, 2, 3, 4, 5, 6, 7, 8, 9],
 [10, 11, 12, 13, 14, 15, 16, 17, 18, 19],
 [20, 21, 22, 23, 24, 25, 26, 27, 28]]

In [11]:
for item in tqdm(pages_and_texts):
    item["sentence_chunks"] = split_list(
        input_list=item["sentences"],
        slice_size=num_sentence_chunk_size
    )

    item["num_chunks"] = len(item["sentence_chunks"])

  0%|          | 0/1208 [00:00<?, ?it/s]

In [12]:
random.sample(pages_and_texts, k=3)

[{'page_number': 1053,
  'page_char_count': 1649,
  'page_word_count': 300,
  'page_sentence_count_raw': 16,
  'page_token_count': 412.25,
  'text': 'The benefits of this kind of diet include an emphasis on whole,  unprocessed foods and a de-emphasis of refined carbohydrates,  such as white flour, white bread, and white sugar. However, there  are a number of downsides. Typically, the first two weeks allow  for only 20 grams of carbs per day, which can be dangerously low.  In addition, dieters using the low-carb approach tend to consume  twice as many saturated fats as people on a diet high in healthy  carbohydrates. Low-carb diets are also associated with a higher  energy intake, and the notion that “calories don’t count,” which  is prevalent in this kind of diet, is not supported by scientific  evidence.8  The Macrobiotic Diet  The macrobiotic diet is part of a health and wellness regimen based  in Eastern philosophy. It combines certain tenets of Zen Buddhism  with a vegetarian diet 

In [13]:
df = pd.DataFrame(pages_and_texts)

df.describe().round(2)

,page_number,page_char_count,page_word_count,page_sentence_count_raw,page_token_count,page_sentence_count_spacy,num_chunks
count,1208.00,1208.00,1208.00,1208.00,1208.00,1208.00,1208.00
mean,562.50,1148.00,199.50,10.52,287.00,10.32,1.53
std,348.86,560.38,95.83,6.55,140.10,6.30,0.64
min,-41.00,0.00,1.00,1.00,0.00,0.00,0.00
25%,260.75,762.00,134.00,5.00,190.50,5.00,1.00
50%,562.50,1231.50,216.00,10.00,307.88,10.00,1.00
75%,864.25,1603.50,272.00,15.00,400.88,15.00,2.00
max,1166.00,2308.00,430.00,39.00,577.00,28.00,3.00


In [19]:
import re

pages_and_chunks = []
for item in tqdm(pages_and_texts):
    for chunk in item["sentence_chunks"]:
        chunk_dict = {}

        chunk_dict["page_number"] = item["page_number"]

        joined_sentence_chunk = "".join(chunk).replace("  ", " ").strip()
        joined_sentence_chunk = re.sub(r'\.([A-Z])', r'. \1', joined_sentence_chunk)


        chunk_dict["sentence_chunk"] = joined_sentence_chunk

        chunk_dict["chunk_char_count"] = len(joined_sentence_chunk)
        chunk_dict["chunk_word_count"] = len([word for word in joined_sentence_chunk.split(" ")])
        chunk_dict["chunk_token_count"] = len(joined_sentence_chunk) / 4

        pages_and_chunks.append(chunk_dict)

len(pages_and_chunks)

  0%|          | 0/1208 [00:00<?, ?it/s]

1843

In [20]:
df = pd.DataFrame(pages_and_chunks)

df.describe().round(2)

,page_number,chunk_char_count,chunk_word_count,chunk_token_count
count,1843.00,1843.00,1843.00,1843.00
mean,583.38,734.10,112.74,183.52
std,347.79,447.51,71.24,111.88
min,-41.00,12.00,3.00,3.00
25%,280.50,315.00,45.00,78.75
50%,586.00,745.00,115.00,186.25
75%,890.00,1118.00,173.00,279.50
max,1166.00,1830.00,297.00,457.50


In [21]:
df.head()

,page_number,sentence_chunk,chunk_char_count,chunk_word_count,chunk_token_count
0,-41,Human Nutrition: 2020 Edition,29,4,7.25
1,-39,Human Nutrition: 2020 Edition UNIVERSITY OF HA...,308,42,77.00
2,-38,Human Nutrition: 2020 Edition by University of...,210,30,52.50
3,-37,Contents Preface University of Hawai‘i at Māno...,766,116,191.50
4,-36,Lifestyles and Nutrition University of Hawai‘i...,941,144,235.25


In [31]:
min_token_length = 30
for row in df[df["chunk_token_count"] <= min_token_length].sample(5).iterrows():
    print(f'Chunk token count: {row[1]["chunk_token_count"]} | Text: {row[1]["sentence_chunk"]}')

Chunk token count: 17.5 | Text: The Obesity Myth. Gotham Books. Calories In Versus Calories Out | 1069
Chunk token count: 11.25 | Text: Accessed March 17, 2011. 212 | Water Concerns
Chunk token count: 21.0 | Text: Updated September 2003. Accessed November 28,2017. Discovering Nutrition Facts | 735
Chunk token count: 29.0 | Text: 2010). EH. Net Encyclopedia. http://eh.net/?s=History+of+Food+and+Drug+Regulatio Protecting the Public Health | 1011
Chunk token count: 26.25 | Text: Snowdon W, Osborn T. (2003). Coconut: It’s role in health. Secretariat of the Pacific. 292 | Introduction


In [32]:
pages_and_chunks_over_min_token_len = df[df["chunk_token_count"] > min_token_length].to_dict(orient="records")
pages_and_chunks_over_min_token_len[:2]

[{'page_number': -39,
  'sentence_chunk': 'Human Nutrition: 2020 Edition UNIVERSITY OF HAWAI‘I AT MĀNOA FOOD SCIENCE AND HUMAN NUTRITION PROGRAM ALAN TITCHENAL, SKYLAR HARA, NOEMI ARCEO CAACBAY, WILLIAM MEINKE-LAU, YA-YUN YANG, MARIE KAINOA FIALKOWSKI REVILLA, JENNIFER DRAPER, GEMADY LANGFELDER, CHERYL GIBBY, CHYNA NICOLE CHUN, AND ALLISON CALABRESE',
  'chunk_char_count': 308,
  'chunk_word_count': 42,
  'chunk_token_count': 77.0},
 {'page_number': -38,
  'sentence_chunk': 'Human Nutrition: 2020 Edition by University of Hawai‘i at Mānoa Food Science and Human Nutrition Program is licensed under a Creative Commons Attribution 4.0 International License, except where otherwise noted.',
  'chunk_char_count': 210,
  'chunk_word_count': 30,
  'chunk_token_count': 52.5}]

In [33]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    model_name_or_path="all-mpnet-base-v2",
    device="mps"
)

sentences = [
    "I would like to play football one day.",
    "My favourite player is Lionel Messi.",
    "It's raining"
]

embeddings = embedding_model.encode(sentences)
embeddings_dict = dict(zip(sentences, embeddings))

for sentence, embedding in embeddings_dict.items():
    print(f"Sentence: {sentence}")
    print(f"Embedding: {embedding}")
    print(" ")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

/Users/nickol/My Fucking Stuff/machine learning/code/local-rag/venv/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Sentence: I would like to play football one day.
Embedding: [-4.21419330e-02  8.62089768e-02 -3.47225778e-02  9.94580239e-03
  6.28689751e-02 -1.68651026e-02 -3.60720754e-02 -1.45378495e-02
  2.88053527e-02  1.04879905e-02  7.16915801e-02 -3.94282751e-02
 -4.30925600e-02  2.41575334e-02  5.15052825e-02  1.35611277e-03
 -2.97665242e-02 -7.42660165e-02 -1.54746436e-02 -4.15016115e-02
 -2.40872335e-02  5.88509403e-02 -8.65249988e-03 -1.57541148e-02
 -4.44777086e-02 -2.39493456e-02  3.70281339e-02 -2.45251395e-02
 -6.23329692e-02  9.17681977e-02 -1.57118437e-03 -4.42774035e-02
  1.71439722e-02 -5.02635874e-02  1.39522149e-06 -4.76675807e-03
 -1.55427819e-02 -6.80074021e-02  2.56581232e-02  6.76748226e-04
  5.12095774e-03 -4.12139669e-02  2.51225824e-03  2.05397932e-03
 -2.02593692e-02  3.90214063e-02  5.24575971e-02 -6.80624023e-02
 -4.66630235e-02  2.69907787e-02 -5.38984127e-03  1.95700992e-02
 -7.44688208e-04 -1.64694656e-02 -2.20816210e-02  3.45719792e-02
 -5.52991070e-02  9.25534591e-

In [34]:
embeddings.shape

(3, 768)

In [39]:
%%time 

for item in tqdm(pages_and_chunks_over_min_token_len):
    item["embedding"] = embedding_model.encode(item["sentence_chunk"])

  0%|          | 0/1680 [00:00<?, ?it/s]

CPU times: user 47.8 s, sys: 10.4 s, total: 58.3 s
Wall time: 1min 20s


In [40]:
text_chunks = [item["sentence_chunk"] for item in pages_and_chunks_over_min_token_len]

len(text_chunks)

1680

In [41]:
random.sample(text_chunks, k=1)

['Another is that combining low- and high-GI foods changes the GI for the meal. Also, some nutrient-dense foods have higher GIs than less nutritious food. (For instance, oatmeal has a higher GI than chocolate because the fat content of chocolate is higher.)Lastly, meats and fats do not have a GI since they do not contain carbohydrates. 250 | Digestion and Absorption of Carbohydrates']

In [43]:
%%time

text_chunk_embeddings = embedding_model.encode(
    text_chunks,
    batch_size=64,
    convert_to_tensor=True
)

text_chunk_embeddings.shape

CPU times: user 2.46 s, sys: 3 s, total: 5.46 s
Wall time: 37.7 s


torch.Size([1680, 768])

In [44]:
text_chunks_and_embeddings_df = pd.DataFrame(pages_and_chunks_over_min_token_len)
embeddings_df_save_path = "text_chunks_and_embeddings_df.csv"
text_chunks_and_embeddings_df.to_csv(embeddings_df_save_path, index=False)

In [45]:
text_chunks_and_embeddings_df_load = pd.read_csv(embeddings_df_save_path)
text_chunks_and_embeddings_df_load.head()

,page_number,sentence_chunk,chunk_char_count,chunk_word_count,chunk_token_count,embedding
0,-39,Human Nutrition: 2020 Edition UNIVERSITY OF HA...,308,42,77.00,[ 6.74242303e-02 9.02281702e-02 -5.09549305e-...
1,-38,Human Nutrition: 2020 Edition by University of...,210,30,52.50,[ 5.52156046e-02 5.92139922e-02 -1.66167002e-...
2,-37,Contents Preface University of Hawai‘i at Māno...,766,116,191.50,[ 2.79801544e-02 3.39813754e-02 -2.06426661e-...
3,-36,Lifestyles and Nutrition University of Hawai‘i...,941,144,235.25,[ 6.82566613e-02 3.81275080e-02 -8.46853200e-...
4,-35,The Cardiovascular System University of Hawai‘...,998,152,249.50,[ 3.30264308e-02 -8.49766564e-03 9.57161095e-...


In [47]:
text_chunks_and_embeddings_df_load["embedding"]

0       [ 6.74242303e-02  9.02281702e-02 -5.09549305e-...
1       [ 5.52156046e-02  5.92139922e-02 -1.66167002e-...
2       [ 2.79801544e-02  3.39813754e-02 -2.06426661e-...
3       [ 6.82566613e-02  3.81275080e-02 -8.46853200e-...
4       [ 3.30264308e-02 -8.49766564e-03  9.57161095e-...
                              ...                        
1675    [ 1.85622703e-02 -1.64277274e-02 -1.27045512e-...
1676    [ 3.34721096e-02 -5.70440777e-02  1.51489452e-...
1677    [ 7.70515651e-02  9.78556927e-03 -1.21817468e-...
1678    [ 1.03045136e-01 -1.64702386e-02  8.26848485e-...
1679    [ 8.63773674e-02 -1.25358859e-02 -1.12746637e-...
Name: embedding, Length: 1680, dtype: object

In [49]:
str_emb = text_chunks_and_embeddings_df_load["embedding"][0]

str_emb

'[ 6.74242303e-02  9.02281702e-02 -5.09549305e-03 -3.17545645e-02\n  7.39082247e-02  3.51976342e-02 -1.97986476e-02  4.67692316e-02\n  5.35727292e-02  5.01227845e-03  3.33928838e-02 -1.62215461e-03\n  1.76080819e-02  3.62652913e-02 -3.16662947e-04 -1.07117947e-02\n  1.54257547e-02  2.62176339e-02  2.77662324e-03  3.64942662e-02\n -4.44109626e-02  1.89362112e-02  4.90118116e-02  1.64020602e-02\n -4.85782735e-02  3.18290503e-03  2.72992756e-02 -2.04757787e-03\n -1.22828595e-02 -7.28049502e-02  1.20446039e-02  1.07300067e-02\n  2.10001529e-03 -8.17773119e-02  2.67830183e-06 -1.81427989e-02\n -1.20802736e-02  2.47174520e-02 -6.27467185e-02  7.35438466e-02\n  2.21624505e-02 -3.28767709e-02 -1.80095714e-02  2.22952366e-02\n  5.61365113e-02  1.79513043e-03  5.25931865e-02 -3.31747695e-03\n -8.33875965e-03 -1.06284656e-02  2.31917971e-03 -2.23934688e-02\n -1.53012108e-02 -9.93055850e-03  4.65322286e-02  3.57469432e-02\n -2.54760254e-02  2.63694208e-02  3.74908559e-03 -3.82680483e-02\n  2.58325

In [52]:
import numpy as np

np.fromstring(str_emb.strip("[]"), sep=" ")

array([ 6.74242303e-02,  9.02281702e-02, -5.09549305e-03, -3.17545645e-02,
        7.39082247e-02,  3.51976342e-02, -1.97986476e-02,  4.67692316e-02,
        5.35727292e-02,  5.01227845e-03,  3.33928838e-02, -1.62215461e-03,
        1.76080819e-02,  3.62652913e-02, -3.16662947e-04, -1.07117947e-02,
        1.54257547e-02,  2.62176339e-02,  2.77662324e-03,  3.64942662e-02,
       -4.44109626e-02,  1.89362112e-02,  4.90118116e-02,  1.64020602e-02,
       -4.85782735e-02,  3.18290503e-03,  2.72992756e-02, -2.04757787e-03,
       -1.22828595e-02, -7.28049502e-02,  1.20446039e-02,  1.07300067e-02,
        2.10001529e-03, -8.17773119e-02,  2.67830183e-06, -1.81427989e-02,
       -1.20802736e-02,  2.47174520e-02, -6.27467185e-02,  7.35438466e-02,
        2.21624505e-02, -3.28767709e-02, -1.80095714e-02,  2.22952366e-02,
        5.61365113e-02,  1.79513043e-03,  5.25931865e-02, -3.31747695e-03,
       -8.33875965e-03, -1.06284656e-02,  2.31917971e-03, -2.23934688e-02,
       -1.53012108e-02, -

In [ ]:
text_chunks_and_embeddings_df_load["embedding"] = text_chunks_and_embeddings_df_load["embedding"].apply(
    lambda x: np.fromstring(x.strip("[]"), sep=" ")
)

In [65]:
embeddings = np.stack(text_chunks_and_embeddings_df_load["embedding"].tolist(), axis=0)

embeddings.shape

(1680, 768)

In [66]:
import torch 

embeddings = torch.tensor(embeddings, dtype=torch.float32).to("mps")

embeddings.shape

torch.Size([1680, 768])

In [67]:
embeddings.device

device(type='mps', index=0)

In [61]:
from sentence_transformers import util, SentenceTransformer

embedding_model = SentenceTransformer(model_name_or_path="all-mpnet-base-v2",
                                      device="mps")

/Users/nickol/My Fucking Stuff/machine learning/code/local-rag/venv/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [75]:
query = "macronutrient functions"

print(f"Query: {query}")

query_embedding = embedding_model.encode(query, convert_to_tensor=True)

from time import perf_counter as timer

start_time = timer()
dot_scores = util.dot_score(a=query_embedding, b=embeddings)
end_time = timer()

print(f"[INFO] Time taken to get scores on {len(embeddings)} embeddings: {end_time - start_time:.5f} seconds.")

top_results_dot_product = torch.topk(dot_scores, k=5)
top_results_dot_product


Query: macronutrient functions
[INFO] Time taken to get scores on 1680 embeddings: 0.00135 seconds.


torch.return_types.topk(
values=tensor([[0.6843, 0.6717, 0.6517, 0.6493, 0.6478]], device='mps:0'),
indices=tensor([[42, 47, 46, 51, 41]], device='mps:0'))

In [81]:
pages_and_texts[42]

{'page_number': 1,
 'page_char_count': 93,
 'page_word_count': 21,
 'page_sentence_count_raw': 3,
 'page_token_count': 23.25,
 'text': 'PART I  CHAPTER 1. BASIC  CONCEPTS IN NUTRITION  Chapter 1. Basic Concepts in Nutrition  |  1',
 'sentences': ['PART I  CHAPTER 1.',
  'BASIC  CONCEPTS IN NUTRITION  Chapter 1.',
  'Basic Concepts in Nutrition  |  1'],
 'page_sentence_count_spacy': 3,
 'sentence_chunks': [['PART I  CHAPTER 1.',
   'BASIC  CONCEPTS IN NUTRITION  Chapter 1.',
   'Basic Concepts in Nutrition  |  1']],
 'num_chunks': 1}